# UK 2016 Trafik Kazası Analizi

**Veri seti:** [UK 2016 Road Safety Data — Kaggle](https://www.kaggle.com/datasets/bluehorseshoe/uk-2016-road-safety-data)  
**Kaynak:** UK Department for Transport  

## Araştırma Soruları
1. Günün hangi saatlerinde ve haftanın hangi günlerinde kazalar en yoğun?
2. Hava koşulları kaza şiddetini etkiliyor mu?
3. Hangi araç türleri en ölümcül kazalara karışıyor?
4. Kaza şiddeti ile saat arasında nasıl bir ilişki var?

---

## İçerik
1. Kütüphaneler ve veri yükleme
2. Veri temizleme ve ön işleme
3. Genel bakış — saat, gün, şiddet dağılımı
4. Hava koşulları analizi
5. Saat × Şiddet ısı haritası
6. Araç türü analizi (merge)
7. Ana bulgular ve sonuç

## 1. Kütüphaneler ve Veri Yükleme

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Kütüphaneler yüklendi.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

accidents  = pd.read_csv('/content/drive/MyDrive/accidents_2016.csv',  low_memory=False)
casualties = pd.read_csv('/content/drive/MyDrive/casualties_2016.csv', low_memory=False)
vehicles   = pd.read_csv('/content/drive/MyDrive/vehicles_2016.csv',   low_memory=False)

print(f'Kazalar   : {accidents.shape[0]:>7,} satır, {accidents.shape[1]} sütun')
print(f'Yaralılar : {casualties.shape[0]:>7,} satır, {casualties.shape[1]} sütun')
print(f'Araçlar   : {vehicles.shape[0]:>7,} satır, {vehicles.shape[1]} sütun')

## 2. Veri Temizleme ve Ön İşleme

In [ ]:
# Tarih ve saat sütunlarını parse et
accidents['Date']     = pd.to_datetime(accidents['Date'], dayfirst=True)
accidents['Hour']     = pd.to_datetime(accidents['Time'], format='%H:%M', errors='coerce').dt.hour
accidents['Month']    = accidents['Date'].dt.month
accidents['Day_Name'] = accidents['Date'].dt.day_name()

# Kaza şiddeti etiketleri
severity_map = {1: 'Ölümlü', 2: 'Ağır Yaralı', 3: 'Hafif Yaralı'}
accidents['Severity_Label'] = accidents['Accident_Severity'].map(severity_map)

# Hava koşulu etiketleri (UK DfT kodları)
hava_map = {
    1: 'Açık',
    2: 'Yağmurlu',
    3: 'Karlı',
    4: 'Açık (Rüzgarlı)',
    5: 'Yağmurlu (Rüzgarlı)',
    6: 'Karlı (Rüzgarlı)',
    7: 'Sisli',
    8: 'Diğer',
    9: 'Bilinmiyor',
   -1: 'Veri Eksik'
}
accidents['Weather_Label'] = accidents['Weather_Conditions'].map(hava_map)

# Araç türü etiketleri
arac_map = {
    1: 'Bisiklet', 2: 'Motosiklet (≤50cc)', 3: 'Motosiklet (≤125cc)',
    4: 'Motosiklet (125-500cc)', 5: 'Motosiklet (>500cc)', 8: 'Taksi',
    9: 'Otomobil', 10: 'Minibüs', 11: 'Otobüs', 16: 'At',
    17: 'Tarım Aracı', 19: 'Hafif Ticari (≤3.5t)',
    20: 'Ağır Ticari (3.5-7.5t)', 21: 'Ağır Ticari (>7.5t)',
    22: 'Engelli Scooter', 90: 'Diğer', 98: 'Ticari (Bilinmiyor)', 99: 'Bilinmiyor'
}
vehicles['Vehicle_Label'] = vehicles['Vehicle_Type'].map(arac_map)

# Eksik değer kontrolü
print('--- Eksik Değerler ---')
eksik = accidents[['Hour', 'Weather_Label', 'Severity_Label']].isnull().sum()
print(eksik[eksik > 0] if eksik.sum() > 0 else 'Eksik değer yok.')
print(f'\nTemizleme tamamlandı. Toplam {len(accidents):,} kaza kaydı hazır.')

## 3. Genel Bakış — Saat, Gün, Şiddet Dağılımı

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('UK 2016 Trafik Kazaları — Genel Bakış', fontsize=14, fontweight='bold')

# 1. Saate göre kaza sayısı
saat_kazalari = accidents['Hour'].value_counts().sort_index()
axes[0,0].bar(saat_kazalari.index, saat_kazalari.values, color='steelblue', alpha=0.85)
axes[0,0].set_title('Saate Göre Kaza Sayısı')
axes[0,0].set_xlabel('Saat')
axes[0,0].set_ylabel('Kaza Sayısı')
axes[0,0].axvline(x=8,  color='red', linestyle='--', alpha=0.5, label='Sabah rush hour')
axes[0,0].axvline(x=17, color='orange', linestyle='--', alpha=0.5, label='Akşam rush hour')
axes[0,0].legend(fontsize=8)

# 2. Güne göre kaza sayısı
gun_sirasi = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
gun_etiket = ['Pzt','Sal','Çar','Per','Cum','Cmt','Paz']
gun_kazalari = accidents['Day_Name'].value_counts().reindex(gun_sirasi)
axes[0,1].bar(range(7), gun_kazalari.values, color='coral', alpha=0.85)
axes[0,1].set_xticks(range(7))
axes[0,1].set_xticklabels(gun_etiket)
axes[0,1].set_title('Güne Göre Kaza Sayısı')
axes[0,1].set_ylabel('Kaza Sayısı')

# 3. Kaza şiddeti pasta grafiği
severity_counts = accidents['Severity_Label'].value_counts()
axes[1,0].pie(severity_counts.values, labels=severity_counts.index,
              autopct='%1.1f%%', colors=['#e74c3c','#e67e22','#3498db'], startangle=90)
axes[1,0].set_title('Kaza Şiddeti Dağılımı')

# 4. Hava koşullarına göre kaza sayısı
hava = accidents['Weather_Label'].value_counts().head(6)
axes[1,1].barh(hava.index, hava.values, color='mediumseagreen', alpha=0.85)
axes[1,1].set_title('Hava Koşullarına Göre Kaza Sayısı')
axes[1,1].set_xlabel('Kaza Sayısı')

plt.tight_layout()
plt.savefig('genel_bakis.png', dpi=150, bbox_inches='tight')
plt.show()

**Gözlemler:**
- Günde iki belirgin tepe: sabah 08:00 ve akşam 17:00 — klasik **rush hour** etkisi.
- **Cuma** en tehlikeli gün (~22.500 kaza); **Pazar** en güvenli (~15.000 kaza).
- Kazaların **%82.9'u hafif yaralıyla** sonuçlanıyor; ölümlü kazalar sadece %1.2.

## 4. Hava Koşulları Analizi

In [ ]:
# Hava koşulu × kaza şiddeti çapraz tablosu
hava_severity = pd.crosstab(
    accidents['Weather_Label'],
    accidents['Severity_Label'],
    normalize='index'
).mul(100).round(1)

# Anlamsız kategorileri çıkar
hava_severity = hava_severity.drop(
    index=['Bilinmiyor', 'Veri Eksik', 'Diğer'], errors='ignore'
)
hava_severity_sorted = hava_severity.sort_values('Ölümlü', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('UK 2016 — Hava Koşullarına Göre Kaza Şiddeti', fontsize=14, fontweight='bold')

# Stacked bar
hava_severity_sorted[['Ölümlü', 'Ağır Yaralı', 'Hafif Yaralı']].plot(
    kind='barh', stacked=True, ax=axes[0],
    color=['#e74c3c', '#e67e22', '#3498db'], alpha=0.85
)
axes[0].set_title('Hava Koşuluna Göre Kaza Şiddeti (%)')
axes[0].set_xlabel('Yüzde (%)')
axes[0].set_ylabel('')
axes[0].legend(loc='lower right', fontsize=9)

# Ölümlü oran sıralaması
olumlu_oran = pd.crosstab(
    accidents['Weather_Label'], accidents['Severity_Label'], normalize='index'
).mul(100).round(2)
olumlu_sorted = olumlu_oran['Ölümlü'].drop(
    index=['Bilinmiyor','Veri Eksik','Diğer'], errors='ignore'
).sort_values()

colors = ['#e74c3c' if x > 1.5 else '#e67e22' if x > 1.0 else '#3498db'
          for x in olumlu_sorted]
axes[1].barh(olumlu_sorted.index, olumlu_sorted.values, color=colors, alpha=0.85)
axes[1].axvline(x=olumlu_sorted.mean(), color='black',
                linestyle='--', alpha=0.5, label=f'Ort. {olumlu_sorted.mean():.2f}%')
axes[1].set_title('Hava Koşuluna Göre Ölümlü Kaza Oranı (%)')
axes[1].set_xlabel('Ölümlü Kaza Oranı (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('hava_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nÖlümlü kaza oranı sıralaması (%):')
print(olumlu_sorted.sort_values(ascending=False).to_string())

**Gözlemler:**
- **Karlı hava** en düşük ağır yaralı oranına sahip — insanlar karla kaplandığında çok daha yavaş ve dikkatli sürüyor (**davranışsal adaptasyon**).
- **Açık (rüzgarlı)** ve **sisli** hava en yüksek ölüm oranlarına sahip — bu koşullar kardan daha az "görünür" tehlike olduğu için sürücüler hız azaltmıyor.
- **Açık havada** en fazla kaza oluyor (112K) ama ölüm oranı düşük — yüksek kaza sayısının sebebi açık havada trafik yoğunluğunun da en fazla olması.

## 5. Saat × Şiddet Isı Haritası

In [ ]:
hour_severity = pd.crosstab(
    accidents['Hour'],
    accidents['Severity_Label'],
    normalize='columns'
).mul(100).round(2)

plt.figure(figsize=(8, 10))
sns.heatmap(
    hour_severity[['Ölümlü', 'Ağır Yaralı', 'Hafif Yaralı']],
    cmap='YlOrRd', linewidths=0.3, annot=True, fmt='.1f',
    cbar_kws={'label': '%'}
)
plt.title('Saate ve Şiddete Göre Kaza Yoğunluğu (%)', fontsize=13, fontweight='bold')
plt.xlabel('Kaza Şiddeti')
plt.ylabel('Saat')
plt.tight_layout()
plt.savefig('heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**Gözlemler:**
- Hafif yaralı ve ağır yaralı sabah **08:00** ve akşam **17:00**'de net zirve yapıyor — rush hour etkisi.
- **Ölümlü kazalar** gece **22:00-02:00** arasında orantılı olarak daha yoğun — az trafik, yüksek hız, azalan dikkat bir arada. 
- Gündüz kaza çok oluyor ama gece olunca daha ölümcül.

## 6. Araç Türü Analizi (Merge)

In [ ]:
# Accidents ve vehicles tablolarını Accident_Index üzerinden birleştir
merged = pd.merge(
    accidents[['Accident_Index', 'Severity_Label', 'Hour', 'Day_Name']],
    vehicles[['Accident_Index', 'Vehicle_Label', 'Age_of_Driver', 'Sex_of_Driver']],
    on='Accident_Index',
    how='inner'
)
print(f'Merge sonrası: {len(merged):,} satır')
merged.head(3)

In [ ]:
# Araç türü × kaza şiddeti
arac_severity = pd.crosstab(
    merged['Vehicle_Label'],
    merged['Severity_Label'],
    normalize='index'
).mul(100).round(2)

# Yeterli veri olan araçları filtrele (≥200 kayıt)
arac_counts  = merged['Vehicle_Label'].value_counts()
gecerli      = arac_counts[arac_counts >= 200].index
arac_severity = arac_severity.loc[gecerli]
arac_sorted   = arac_severity.sort_values('Ölümlü', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('UK 2016 — Araç Türüne Göre Kaza Analizi', fontsize=14, fontweight='bold')

# Ölümlü oran bar grafiği
colors = ['#e74c3c' if x > 2 else '#e67e22' if x > 1 else '#3498db'
          for x in arac_sorted['Ölümlü']]
axes[0].barh(arac_sorted.index, arac_sorted['Ölümlü'], color=colors, alpha=0.85)
axes[0].axvline(x=arac_sorted['Ölümlü'].mean(), color='black',
                linestyle='--', alpha=0.5, label='Ortalama')
axes[0].set_title('Araç Türüne Göre Ölümlü Kaza Oranı (%)')
axes[0].set_xlabel('Ölümlü Kaza Oranı (%)')
axes[0].legend()

# Stacked bar
arac_sorted[['Ölümlü', 'Ağır Yaralı', 'Hafif Yaralı']].plot(
    kind='barh', stacked=True, ax=axes[1],
    color=['#e74c3c', '#e67e22', '#3498db'], alpha=0.85
)
axes[1].set_title('Araç Türüne Göre Kaza Şiddeti Dağılımı (%)')
axes[1].set_xlabel('Yüzde (%)')
axes[1].set_ylabel('')
axes[1].legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('arac_analiz.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nÖlümlü kaza oranı sıralaması (%)')
print(arac_severity['Ölümlü'].sort_values(ascending=False).to_string())

**Gözlemler:**
- **Ağır ticari (>7.5t)** %5.26 ile en ölümcül gerçek araç kategorisi — büyük TIR karışınca kaza hafif geçmiyor.
- **Motosiklet içinde 9 kat fark:** ≤50cc %0.42 iken >500cc %3.82. Motor gücü = hız = ölüm riski.
- **Bisiklet sadece %0.58** — yavaş hareket eden araçlar düşük hızda kaza yapıyor, ölümcül olmayan çarpışmalar çoğunlukta.
- **Motosiklet (>500cc)** ağır yaralı oranı (~%40) tüm kategorilerin en yükseği — ya ölüm ya ağır yaralanma, hafif atlatma şansı çok düşük.

> ⚠️ **Not:** Engelli Scooter %5.38 görünüyor ancak kayıt sayısı çok az; bu sonuç istatistiksel olarak güvenilir değil.

## 7. Ana Bulgular ve Sonuç

| # | Bulgu | Detay |
|---|---|---|
| 1 | **En tehlikeli saat: 17:00** | Akşam rush hour zirvesi, 12.000+ kaza |
| 2 | **En tehlikeli gün: Cuma** | ~22.500 kaza; Pazar en güvenli (~15.000) |
| 3 | **Gece kazaları daha ölümcül** | 22:00-02:00 arası ölümlü kaza oranı orantılı yüksek |
| 4 | **Karlı hava paradoksu** | Az kaza, düşük şiddet — sürücüler davranışını adapte ediyor |
| 5 | **Motor gücü = ölüm riski** | Motosiklet ≤50cc %0.42 → >500cc %3.82 (9 kat fark) |

### Öğrenilen Kavramlar

| Kavram | Kullanıldığı yer |
|---|---|
| `pd.merge()` | Accidents + Vehicles tabloları birleştirildi |
| `pd.crosstab(normalize='index')` | Hava/araç × şiddet yüzde tabloları |
| `map()` ile kod→etiket | Numerik kodlar okunabilir kategorilere dönüştürüldü |
| `sns.heatmap()` | Saat × şiddet ilişkisi görselleştirildi |
| Small sample bias | Engelli Scooter / Veri Eksik yorumu |
| Davranışsal adaptasyon | Karlı havada düşük kaza şiddeti |

### Sonraki Adımlar
- Coğrafi analiz: Longitude/Latitude ile kaza yoğunluk haritası (folium)
- Sınıflandırma modeli: Kaza şiddetini tahmin eden ML modeli
- Zaman serisi: Aylık kaza trendi analizi